In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense
)

In [2]:
df = pd.read_csv(
    "../../datasets/EVProject_dataset.csv"
)

In [3]:
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"]
)

df = df.sort_values(
    by="Timestamp"
).reset_index(drop=True)

In [4]:
df["voltage_1h"] = (
    df["Battery_Voltage"].shift(-4)
)

df["voltage_6h"] = (
    df["Battery_Voltage"].shift(-24)
)

df["voltage_24h"] = (
    df["Battery_Voltage"].shift(-96)
)

In [5]:
VOLTAGE_TARGETS = [
    "voltage_1h",
    "voltage_6h",
    "voltage_24h"
]

In [6]:
voltage_df = df.dropna(
    subset=VOLTAGE_TARGETS
).copy()

print(voltage_df.shape)

(148988, 33)


In [7]:
FEATURE_COLUMNS = [
    "SoC",
    "SoH",
    "Battery_Voltage",
    "Battery_Current",
    "Power_Consumption",
    "Driving_Speed"
]

In [8]:
split_idx = int(
    len(voltage_df) * 0.8
)

train_df = voltage_df.iloc[:split_idx]

test_df = voltage_df.iloc[split_idx:]

In [9]:
feature_scaler = MinMaxScaler()

train_df[FEATURE_COLUMNS] = (
    feature_scaler.fit_transform(
        train_df[FEATURE_COLUMNS]
    )
)

test_df[FEATURE_COLUMNS] = (
    feature_scaler.transform(
        test_df[FEATURE_COLUMNS]
    )
)

joblib.dump(
    feature_scaler,
    "voltage_feature_scaler.pkl"
)

['voltage_feature_scaler.pkl']

In [10]:
target_scaler = MinMaxScaler()

train_df[VOLTAGE_TARGETS] = (
    target_scaler.fit_transform(
        train_df[VOLTAGE_TARGETS]
    )
)

test_df[VOLTAGE_TARGETS] = (
    target_scaler.transform(
        test_df[VOLTAGE_TARGETS]
    )
)

joblib.dump(
    target_scaler,
    "voltage_target_scaler.pkl"
)

['voltage_target_scaler.pkl']

In [11]:
X_train = (
    train_df[FEATURE_COLUMNS]
    .values
    .astype("float32")
)

y_train = (
    train_df[VOLTAGE_TARGETS]
    .values
    .astype("float32")
)

X_test = (
    test_df[FEATURE_COLUMNS]
    .values
    .astype("float32")
)

y_test = (
    test_df[VOLTAGE_TARGETS]
    .values
    .astype("float32")
)

In [12]:
WINDOW_SIZE = 24

def create_sequences(
    X,
    y,
    window_size
):

    X_seq = []
    y_seq = []

    for i in range(
        window_size,
        len(X)
    ):

        X_seq.append(
            X[
                i-window_size:i
            ]
        )

        y_seq.append(
            y[i]
        )

    return (
        np.array(
            X_seq,
            dtype=np.float32
        ),
        np.array(
            y_seq,
            dtype=np.float32
        )
    )

In [14]:
X_train_seq, y_train_seq = (
    create_sequences(
        X_train,
        y_train,
        WINDOW_SIZE
    )
)

X_test_seq, y_test_seq = (
    create_sequences(
        X_test,
        y_test,
        WINDOW_SIZE
    )
)

print(X_train_seq.shape)
print(y_train_seq.shape)

(119166, 24, 6)
(119166, 3)


In [15]:
model = Sequential([

    LSTM(
        32,
        input_shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    ),

    Dense(
        16,
        activation="relu"
    ),

    Dense(3)

])

c:\Users\shrey\OneDrive\Desktop\MY_Project\EV_Intelligence_System\Backend\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [16]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [17]:
history = model.fit(
    X_train_seq,
    y_train_seq,
    epochs=5,
    batch_size=32,
    validation_data=(
        X_test_seq,
        y_test_seq
    )
)

Epoch 1/5
3724/3724 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 0.0780 - mae: 0.1940 - val_loss: 0.0767 - val_mae: 0.1892
Epoch 2/5
3724/3724 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - loss: 0.0770 - mae: 0.1922 - val_loss: 0.0755 - val_mae: 0.1858
Epoch 3/5
3724/3724 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 0.0769 - mae: 0.1919 - val_loss: 0.0764 - val_mae: 0.1762
Epoch 4/5
3724/3724 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - loss: 0.0769 - mae: 0.1918 - val_loss: 0.0756 - val_mae: 0.1837
Epoch 5/5
3724/3724 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 0.0768 - mae: 0.1917 - val_loss: 0.0756 - val_mae: 0.1824


In [18]:
model.save(
    "voltage_forecast_model.keras"
)

In [19]:
preds = model.predict(
    X_test_seq[:1000]
)

preds = target_scaler.inverse_transform(
    preds
)

actual = target_scaler.inverse_transform(
    y_test_seq[:1000]
)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [20]:
from sklearn.metrics import mean_squared_error
import numpy as np

for i, horizon in enumerate(
    ["1H", "6H", "24H"]
):

    rmse = np.sqrt(
        mean_squared_error(
            actual[:, i],
            preds[:, i]
        )
    )

    print(
        horizon,
        rmse
    )

1H 55.284705675626284
6H 54.785929278191496
24H 55.963646124068795
